# Mask R-CNN for Hot3D In-Hand Object Segmentation

This notebook implements the complete workflow for training and using Mask R-CNN models for in-hand object segmentation on the Hot3D dataset:

1. **Data Preparation**: Loading and processing Hot3D sequences
2. **Model Training**: Fine-tuning Mask R-CNN for in-hand object segmentation
3. **Evaluation**: Testing the model on held-out sequences
4. **Visualization**: Visualizing the segmentation results

All models, results and visualizations will be stored in an organized project structure.

## 1. Setup and Configuration

In [1]:
import os
import sys
import numpy as np
import torch
import torchvision
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.models.detection.mask_rcnn import MaskRCNN_ResNet50_FPN_Weights
from torchvision.transforms import functional as F
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from tqdm import tqdm
import time
import json
import random
from torch.utils.data import Dataset, DataLoader
import cv2
from sklearn.model_selection import train_test_split

# Configure project directory structure
home = os.path.expanduser("/storage/aspoto")
hot3d_dataset_path = os.path.join(home, "hot3d/hot3d/dataset")
project_dir = os.path.join(home, "hot3d/hot3d/mask_rcnn_project")

# Create project directories
models_dir = os.path.join(project_dir, "models")
results_dir = os.path.join(project_dir, "results")
visualizations_dir = os.path.join(project_dir, "visualizations")

for dir_path in [project_dir, models_dir, results_dir, visualizations_dir]:
    os.makedirs(dir_path, exist_ok=True)

# Add Hot3D to the path
sys.path.append('/storage/aspoto/hot3d/hot3d')

# Import Hot3D libraries
from dataset_api import Hot3dDataProvider
from data_loaders.loader_object_library import load_object_library
from projectaria_tools.core.sensor_data import TimeDomain, TimeQueryOptions
from projectaria_tools.core.stream_id import StreamId
from data_loaders.loader_hand_poses import LEFT_HAND_INDEX, RIGHT_HAND_INDEX

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Check available device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Define visualization colors
BLUE_MASK_COLOR = [0, 110, 255]  # Blue for masks
GREEN_CONTOUR_COLOR = [0, 255, 50]  # Green for ground truth contours
MASK_ALPHA = 0.6  # Mask opacity

# Normalized colors for matplotlib (range 0-1)
BLUE_MASK_COLOR_NORM = [c/255 for c in BLUE_MASK_COLOR]
GREEN_CONTOUR_COLOR_NORM = [c/255 for c in GREEN_CONTOUR_COLOR]

Using device: cuda


## 2. Dataset Preparation

We'll create a custom dataset class to load sequences from the Hot3D dataset.

In [2]:
class Hot3DHandObjectDataset(Dataset):
    """
    Dataset for in-hand objects in the Hot3D dataset
    """
    def __init__(self, 
                 sequence_paths, 
                 transform=None, 
                 num_samples_per_sequence=20,
                 filter_empty=True,
                 include_hands=False):
        """
        Args:
            sequence_paths (list): List of paths to Hot3D sequences
            transform (callable, optional): Optional transform to apply to images
            num_samples_per_sequence (int): Number of samples to extract from each sequence
            filter_empty (bool): Whether to filter frames without objects
            include_hands (bool): Whether to include hands as objects to detect
        """
        self.sequence_paths = sequence_paths
        self.transform = transform
        self.num_samples_per_sequence = num_samples_per_sequence
        self.filter_empty = filter_empty
        self.include_hands = include_hands
        
        # Prepare dataset
        self.samples = []
        self._prepare_dataset()
        
        print(f"Dataset prepared with {len(self.samples)} samples")
        
    def _prepare_dataset(self):
        """Prepare the dataset by extracting samples from sequences"""
        # Path to object library
        object_library_path = os.path.join(hot3d_dataset_path, "assets")
        
        # Load object library
        object_library = load_object_library(object_library_folderpath=object_library_path)
        
        for sequence_path in tqdm(self.sequence_paths, desc="Loading sequences"):
            # Initialize data provider
            try:
                hot3d_data_provider = Hot3dDataProvider(
                    sequence_folder=sequence_path,
                    object_library=object_library
                )
                
                # Get device data provider
                device_data_provider = hot3d_data_provider.device_data_provider
                
                # Get object bounding box data provider
                object_box2d_data_provider = hot3d_data_provider.object_box2d_data_provider
                
                # Get hand bounding box data provider
                hand_box2d_data_provider = hot3d_data_provider.hand_box2d_data_provider
                
                # Get timestamps
                timestamps = device_data_provider.get_sequence_timestamps()
                
                # Get image stream IDs
                image_stream_ids = device_data_provider.get_image_stream_ids()
                
                # Select RGB stream if available, otherwise use SLAM-LEFT
                if StreamId("214-1") in image_stream_ids:
                    stream_id = StreamId("214-1")  # RGB
                else:
                    stream_id = StreamId("1201-1")  # SLAM-LEFT
                
                # Sample timestamps
                sampled_indices = np.linspace(
                    0, len(timestamps)-1, 
                    min(self.num_samples_per_sequence, len(timestamps)), 
                    dtype=int
                )
                
                for idx in sampled_indices:
                    timestamp_ns = timestamps[idx]
                    
                    # Get image data
                    image_data = device_data_provider.get_image(timestamp_ns, stream_id)
                    
                    # Initialize targets dictionary
                    targets = {'boxes': [], 'labels': [], 'masks': []}
                    
                    if object_box2d_data_provider is not None:
                        box2d_collection_with_dt = object_box2d_data_provider.get_bbox_at_timestamp(
                            stream_id=stream_id,
                            timestamp_ns=timestamp_ns,
                            time_query_options=TimeQueryOptions.CLOSEST,
                            time_domain=TimeDomain.TIME_CODE,
                        )
                        
                        if box2d_collection_with_dt is not None:
                            # Extract object bounding boxes
                            for obj_uid in box2d_collection_with_dt.box2d_collection.object_uid_list:
                                bbox_data = box2d_collection_with_dt.box2d_collection.box2ds[obj_uid]
                                if bbox_data.box2d is not None:
                                    # Convert box2d to [x1, y1, x2, y2] format
                                    bb = bbox_data.box2d
                                    box = np.array([bb.left, bb.top, bb.left + bb.width, bb.top + bb.height])
                                    
                                    # Add box to annotations
                                    targets['boxes'].append(box)
                                    targets['labels'].append(1)  # 1 = object (in COCO, 0 is background)
                                    
                                    # Create an approximate mask based on the bounding box
                                    # This is a simplified approach; in a real case, you should use real masks
                                    mask = np.zeros((image_data.shape[0], image_data.shape[1]), dtype=np.uint8)
                                    x1, y1, x2, y2 = box.astype(int)
                                    mask[y1:y2, x1:x2] = 1
                                    targets['masks'].append(mask)
                    
                    # Add hands if requested
                    if self.include_hands and hand_box2d_data_provider is not None:
                        hand_box2d_collection_with_dt = hand_box2d_data_provider.get_bbox_at_timestamp(
                            stream_id=stream_id,
                            timestamp_ns=timestamp_ns,
                            time_query_options=TimeQueryOptions.CLOSEST,
                            time_domain=TimeDomain.TIME_CODE,
                        )
                        
                        if hand_box2d_collection_with_dt is not None:
                            # Extract hand bounding boxes
                            for hand_id in [LEFT_HAND_INDEX, RIGHT_HAND_INDEX]:
                                if hand_id in hand_box2d_collection_with_dt.box2d_collection.box2ds:
                                    bbox_data = hand_box2d_collection_with_dt.box2d_collection.box2ds[hand_id]
                                    if bbox_data.box2d is not None:
                                        # Convert box2d to [x1, y1, x2, y2] format
                                        bb = bbox_data.box2d
                                        box = np.array([bb.left, bb.top, bb.left + bb.width, bb.top + bb.height])
                                        
                                        # Add box to annotations
                                        targets['boxes'].append(box)
                                        targets['labels'].append(2)  # 2 = hand
                                        
                                        # Approximate mask
                                        mask = np.zeros((image_data.shape[0], image_data.shape[1]), dtype=np.uint8)
                                        x1, y1, x2, y2 = box.astype(int)
                                        mask[y1:y2, x1:x2] = 1
                                        targets['masks'].append(mask)
                    
                    # Skip frames without objects if requested
                    if self.filter_empty and len(targets['boxes']) == 0:
                        continue
                    
                    # Convert to tensors
                    if len(targets['boxes']) > 0:
                        targets['boxes'] = torch.tensor(np.array(targets['boxes']), dtype=torch.float32)
                        targets['labels'] = torch.tensor(np.array(targets['labels']), dtype=torch.int64)
                        targets['masks'] = torch.tensor(np.array(targets['masks']), dtype=torch.uint8)
                    
                    # Add sample
                    self.samples.append({
                        'image': image_data,
                        'targets': targets,
                        'sequence': os.path.basename(sequence_path),
                        'timestamp': timestamp_ns
                    })
                    
            except Exception as e:
                print(f"Error loading sequence {sequence_path}: {e}")
    
    def __len__(self):
        """Returns the number of samples in the dataset"""
        return len(self.samples)
    
    def __getitem__(self, idx):
        """Returns a sample from the dataset"""
        sample = self.samples[idx]
        
        image = sample['image']
        targets = sample['targets']
        
        # Convert to tensor if not already
        if not isinstance(image, torch.Tensor):
            image = F.to_tensor(image)
        
        # Apply transformations if available
        if self.transform:
            image, targets = self.transform(image, targets)
        
        return image, targets

## 3. Mask R-CNN Model Handler

Next, we'll create a handler class for Mask R-CNN that will manage training, evaluation, and inference.

In [33]:
class MaskRCNNHandler:
    """
    Simplified and robust Mask R-CNN handler for in-hand object segmentation
    """
    def __init__(self, num_classes=2, pretrained=True):
        """
        Initialize the Mask R-CNN model
        
        Args:
            num_classes (int): Number of classes (including background)
            pretrained (bool): Whether to load the model pre-trained on COCO
        """
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"Using device: {self.device}")
        
        # Initialize model
        if pretrained:
            # Modern approach to load pre-trained weights
            weights = MaskRCNN_ResNet50_FPN_Weights.DEFAULT
            self.model = maskrcnn_resnet50_fpn(weights=weights)
            
            # Modify the number of classes in the box prediction head
            in_features = self.model.roi_heads.box_predictor.cls_score.in_features
            self.model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(
                in_features, num_classes
            )
            
            # Modify the number of classes in the mask prediction head
            in_features_mask = self.model.roi_heads.mask_predictor.conv5_mask.in_channels
            hidden_layer = 256
            self.model.roi_heads.mask_predictor = torchvision.models.detection.mask_rcnn.MaskRCNNPredictor(
                in_features_mask, hidden_layer, num_classes
            )
        else:
            # Initialize from scratch
            self.model = maskrcnn_resnet50_fpn(weights=None, num_classes=num_classes)
        
        self.model.to(self.device)
        
        # Dictionary to map class indices to names
        self.class_names = {
            0: 'background',  # Always 0 in PyTorch
            1: 'object',      # In-hand objects
            2: 'hand'         # Hands (if included)
        }
    
    def train(self, train_dataset, val_dataset=None, num_epochs=10, batch_size=2, lr=0.005):
        """
        Fine-tune the model on a specific dataset
        
        Args:
            train_dataset: Training dataset
            val_dataset: Validation dataset (optional)
            num_epochs (int): Number of epochs
            batch_size (int): Batch size
            lr (float): Learning rate
        """
        # Set model to training mode
        self.model.train()
        
        # Data loader
        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=4,
            collate_fn=self._collate_fn
        )
        
        # Optimizer
        params = [p for p in self.model.parameters() if p.requires_grad]
        optimizer = torch.optim.SGD(
            params, 
            lr=lr,
            momentum=0.9, 
            weight_decay=0.0005
        )
        
        # Learning rate scheduler
        lr_scheduler = torch.optim.lr_scheduler.StepLR(
            optimizer,
            step_size=3,
            gamma=0.1
        )
        
        # Initialize structures to track metrics
        history = {
            'loss': [],
            'lr': [],
            'val_loss': [] if val_dataset else None
        }
        
        print("Starting training...")
        for epoch in range(num_epochs):
            epoch_start = time.time()
            epoch_loss = 0
            
            # Progress bar for each epoch
            with tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}") as pbar:
                for images, targets in pbar:
                    try:
                        # Move to device
                        images = [image.to(self.device) for image in images]
                        targets = [{k: v.to(self.device) if isinstance(v, torch.Tensor) else v 
                                for k, v in t.items()} for t in targets]
                        
                        # Clean gradients
                        optimizer.zero_grad()
                        
                        # Forward pass
                        loss_dict = self.model(images, targets)
                        
                        # Calculate total loss, handling both dict and list cases
                        if isinstance(loss_dict, dict):
                            losses = sum(loss for loss in loss_dict.values())
                        elif isinstance(loss_dict, list):
                            # For cases where the model returns a list
                            # This often happens when there are no valid targets
                            # Create a zero loss tensor that requires gradient
                            losses = torch.tensor(0.0, device=self.device, requires_grad=True)
                            for loss in loss_dict:
                                if isinstance(loss, dict):
                                    losses = losses + sum(l for l in loss.values())
                                elif isinstance(loss, torch.Tensor):
                                    losses = losses + loss
                        else:
                            # If it's already a tensor
                            losses = loss_dict
                        
                        # Backward pass
                        losses.backward()
                        optimizer.step()
                        
                        # Update epoch loss and progress bar
                        loss_value = losses.item()
                        epoch_loss += loss_value
                        pbar.set_postfix({'loss': loss_value})
                    
                    except Exception as e:
                        print(f"Error in batch: {e}")
                        
            # Update scheduler
            lr_scheduler.step()
            
            # Calculate average epoch loss
            avg_loss = epoch_loss / len(train_loader) if len(train_loader) > 0 else float('inf')
            
            # Evaluate on validation dataset (if provided)
            val_loss = float('inf')  # Default value
            if val_dataset:
                try:
                    val_loss = self._evaluate(val_dataset, batch_size)
                    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}, Val Loss: {val_loss:.4f}, "
                        f"LR: {optimizer.param_groups[0]['lr']:.6f}, "
                        f"Time: {time.time() - epoch_start:.2f}s")
                except Exception as e:
                    print(f"Validation error: {e} - skipping validation for this epoch")
            else:
                print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}, "
                      f"LR: {optimizer.param_groups[0]['lr']:.6f}, "
                      f"Time: {time.time() - epoch_start:.2f}s")
            
            # Update training history
            history['loss'].append(avg_loss)
            history['lr'].append(optimizer.param_groups[0]['lr'])
            if val_dataset:
                history['val_loss'].append(val_loss)
            handler.save_model(model_save_path, epoch=epoch)
        
        print("Training completed!")
        return history
    
    def _evaluate(self, dataset, batch_size=2):
        """Evaluate the model on a dataset with tensor type conversion"""
        self.model.eval()
        data_loader = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=False,
            num_workers=4,
            collate_fn=self._collate_fn
        )
        
        total_loss = 0
        num_batches = 0
        
        # Helper function to safely process tensors
        def safe_process_tensor(tensor):
            """Safely process a tensor regardless of type or shape"""
            try:
                # Check if it's a tensor
                if not isinstance(tensor, torch.Tensor):
                    return float(tensor)  # Convert to float if it's a scalar
                    
                # Convert Long tensor to Float if needed
                if tensor.dtype == torch.long:
                    tensor = tensor.float()
                    
                # Handle multi-element tensors
                if tensor.numel() > 1:
                    return tensor.mean().item()
                else:
                    return tensor.item()
            except Exception as e:
                print(f"Error processing tensor: {e}, tensor type: {type(tensor)}, "
                      f"tensor shape: {tensor.shape if hasattr(tensor, 'shape') else 'unknown'}")
                return 0.0  # Return zero for failed processing
        
        with torch.no_grad():
            for images, targets in tqdm(data_loader, desc="Validation"):
                try:
                    images = [image.to(self.device) for image in images]
                    targets = [{k: v.to(self.device) if isinstance(v, torch.Tensor) else v 
                              for k, v in t.items()} for t in targets]
                    
                    # Forward pass
                    loss_dict = self.model(images, targets)
                    
                    # Handle different output types
                    if isinstance(loss_dict, dict):
                        # Sum all loss values in the dictionary
                        batch_loss = sum(safe_process_tensor(v) for v in loss_dict.values())
                    
                    elif isinstance(loss_dict, list):
                        # Process list of loss items
                        batch_loss = 0
                        for loss_item in loss_dict:
                            if isinstance(loss_item, dict):
                                # Sum dictionary values
                                batch_loss += sum(safe_process_tensor(v) for v in loss_item.values())
                            else:
                                # Process single item
                                batch_loss += safe_process_tensor(loss_item)
                    
                    else:
                        # Process single loss
                        batch_loss = safe_process_tensor(loss_dict)
                    
                    total_loss += batch_loss
                    num_batches += 1
                
                except Exception as e:
                    print(f"Error in validation batch: {e}")
                    continue
        
        self.model.train()
        return total_loss / max(num_batches, 1)  # Avoid division by zero
    
    def _collate_fn(self, batch):
        """Custom collate function to handle images of different sizes"""
        images = [item[0] for item in batch]
        targets = [item[1] for item in batch]
        return images, targets
    
    def save_model(self, path, epoch=None):
        """Save model to disk"""
        # Create directory if it doesn't exist
        os.makedirs(os.path.dirname(path), exist_ok=True)
        
        # Add epoch number to filename if provided
        if epoch is not None:
            base, ext = os.path.splitext(path)
            path = f"{base}_epoch_{epoch}{ext}"
            
        # Save model
        torch.save(self.model.state_dict(), path)
        print(f"Model saved to {path}")
        return path
    
    def load_model(self, path):
        """Load model from disk"""
        if not os.path.exists(path):
            print(f"Warning: Model file {path} not found")
            return False
            
        try:
            self.model.load_state_dict(torch.load(path, map_location=self.device))
            self.model.eval()
            print(f"Model loaded from {path}")
            return True
        except Exception as e:
            print(f"Error loading model: {e}")
            return False
    
    def predict(self, image, score_threshold=0.5, hand_filter=True):
        """
        Run inference on an image
        
        Args:
            image: Image as numpy array, PIL Image, or PyTorch tensor
            score_threshold: Minimum confidence threshold
            hand_filter: Whether to filter for in-hand objects
        """
        # Set model to evaluation mode
        self.model.eval()
        
        # Handle different image types
        if isinstance(image, torch.Tensor):
            # Already a tensor, just ensure it's on the right device
            image_tensor = image.to(self.device)
        elif isinstance(image, np.ndarray):
            # Convert numpy array to tensor
            image_tensor = F.to_tensor(image).to(self.device)
        else:
            # Assume it's a PIL Image
            image_tensor = F.to_tensor(image).to(self.device)
        
        # Add batch dimension if needed
        if image_tensor.dim() == 3:
            # Run inference
            with torch.no_grad():
                prediction = self.model([image_tensor])[0]
        else:
            # Already has batch dimension
            with torch.no_grad():
                prediction = self.model(image_tensor)[0]
        
        # Extract results
        boxes = prediction["boxes"].cpu().numpy()
        labels = prediction["labels"].cpu().numpy()
        scores = prediction["scores"].cpu().numpy()
        masks = prediction["masks"].squeeze().cpu().numpy()
        
        # Filter by confidence threshold
        keep = scores >= score_threshold
        boxes = boxes[keep]
        labels = labels[keep]
        scores = scores[keep]
        if len(masks.shape) > 2:
            masks = masks[keep]
        
        # Convert labels to class names
        class_names = [self.class_names.get(label, f"class_{label}") for label in labels]
        
        # Filter for in-hand objects if requested
        if hand_filter:
            # Include only objects (label=1)
            keep_objects = labels == 1
            boxes = boxes[keep_objects]
            labels = labels[keep_objects]
            scores = scores[keep_objects]
            class_names = [class_names[i] for i in range(len(class_names)) if keep_objects[i]]
            if len(masks.shape) > 2:
                masks = masks[keep_objects]
        
        # Create result dictionary
        result = {
            "boxes": boxes,
            "labels": labels,
            "scores": scores,
            "masks": masks,
            "class_names": class_names
        }
        
        return result
    
    def evaluate_model(self, test_dataset, batch_size=2):
        """Evaluate model on test dataset and compute metrics"""
        self.model.eval()
        data_loader = DataLoader(
            test_dataset,
            batch_size=batch_size,
            shuffle=False,
            num_workers=4,
            collate_fn=self._collate_fn
        )
        
        # Metrics
        metrics = {
            'detection_count': [],
            'gt_count': [],
            'sample_metrics': []
        }
        
        with torch.no_grad():
            for images, targets in tqdm(data_loader, desc="Evaluating"):
                try:
                    images = [image.to(self.device) for image in images]
                    
                    # Get predictions
                    predictions = self.model(images)
                    
                    # Calculate metrics for each image
                    for i, (pred, target) in enumerate(zip(predictions, targets)):
                        # Count detections above threshold
                        pred_boxes = pred['boxes'][pred['scores'] > 0.5]
                        pred_count = len(pred_boxes)
                        
                        # Count ground truth objects
                        gt_boxes = target['boxes']
                        gt_count = len(gt_boxes)
                        
                        metrics['detection_count'].append(pred_count)
                        metrics['gt_count'].append(gt_count)
                        
                        # Store individual sample metrics
                        metrics['sample_metrics'].append({
                            'pred_count': pred_count,
                            'gt_count': gt_count,
                            'diff': abs(pred_count - gt_count)
                        })
                except Exception as e:
                    print(f"Error in evaluation batch: {e}")
                    continue
        
        # Calculate average metrics
        if metrics['detection_count']:
            metrics['avg_detection_count'] = np.mean(metrics['detection_count'])
            metrics['avg_gt_count'] = np.mean(metrics['gt_count'])
            metrics['avg_diff'] = np.mean([m['diff'] for m in metrics['sample_metrics']])
            
            # Print summary
            print(f"Evaluation Results:")
            print(f"Total predicted objects: {sum(metrics['detection_count'])}")
            print(f"Total ground truth objects: {sum(metrics['gt_count'])}")
            print(f"Average difference per frame: {metrics['avg_diff']:.2f}")
        else:
            print("No valid evaluation metrics collected")
        
        self.model.train()
        return metrics

## 4. Visualization Functions

Now we'll define functions to visualize the segmentation results.

In [4]:
def visualize_results(image, predictions, score_threshold=0.5, show_boxes=False, show_labels=False):
    """
    Visualize detection results with improved styling - masks only
    
    Args:
        image: Original image (numpy array or PIL Image)
        predictions: Model predictions
        score_threshold: Minimum confidence threshold
        show_boxes: Whether to show bounding boxes (default: False)
        show_labels: Whether to show labels (default: False)
    """
    # Convert image to numpy array if needed
    if isinstance(image, torch.Tensor):
        image_np = image.cpu().numpy().transpose(1, 2, 0)
        # If image has values between 0 and 1, convert to 0-255
        if image_np.max() <= 1.0:
            image_np = (image_np * 255).astype(np.uint8)
    elif isinstance(image, np.ndarray):
        image_np = image.copy()
    else:
        image_np = np.array(image)
    
    # Create figure with dimensions proportional to the image
    height, width = image_np.shape[:2]
    fig_width = 12
    fig_height = fig_width * height / width
    
    fig, ax = plt.subplots(1, figsize=(fig_width, fig_height))
    
    # Show original image
    ax.imshow(image_np)
    
    # Get predictions
    boxes = predictions["boxes"]
    labels = predictions["labels"]
    scores = predictions["scores"]
    class_names = predictions["class_names"]
    masks = predictions["masks"]
    
    # Filter by confidence threshold (if not already done)
    if isinstance(scores, np.ndarray) and len(scores) > 0 and score_threshold > 0:
        keep = scores >= score_threshold
        boxes = boxes[keep]
        labels = labels[keep]
        scores = scores[keep]
        class_names = [class_names[i] for i in range(len(class_names)) if keep[i]]
        if len(masks.shape) > 2:
            masks = masks[keep]
    
    # Normalized colors for matplotlib (0-1 range)
    blue_color_norm = BLUE_MASK_COLOR_NORM.copy()
    
    # Visualize results
    if len(boxes) > 0:
        # Prepare empty image for combined masks
        combined_mask = np.zeros((height, width, 4), dtype=np.uint8)
        
        # For each detected object
        for i, (box, label, score, name) in enumerate(zip(boxes, labels, scores, class_names)):
            # Draw bounding box if requested
            if show_boxes:
                x1, y1, x2, y2 = box.astype(int)
                rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, 
                                    edgecolor=blue_color_norm, linewidth=2)
                ax.add_patch(rect)
            
            # Add label with score if requested
            if show_labels:
                x1, y1 = box.astype(int)[:2]
                label_text = f'{name}: {score:.2f}'
                ax.text(x1, y1-10, label_text, color='white', fontsize=12,
                       bbox=dict(facecolor=blue_color_norm, alpha=0.8, pad=2, edgecolor='none'))
            
            # Overlay mask with transparency
            if len(masks.shape) > 2:
                mask = masks[i]
                mask_binary = (mask > 0.5).astype(np.uint8)
                
                # Create RGBA mask
                r, g, b = BLUE_MASK_COLOR  # Use 0-255 values for image mask
                mask_color = np.zeros((height, width, 4), dtype=np.uint8)
                mask_color[mask_binary > 0] = [r, g, b, int(255 * MASK_ALPHA)]
                
                # Combine with existing mask
                # Use maximum to avoid overlaps
                combined_mask = np.maximum(combined_mask, mask_color)
        
        # Overlay the combined mask
        mask_img = Image.fromarray(combined_mask, 'RGBA')
        ax.imshow(mask_img)
    
    ax.set_title(f"In-hand Object Segmentation: {len(boxes)} objects detected", fontsize=14)
    ax.axis('off')
    plt.tight_layout(pad=0)
    
    return fig

def create_composite_figure(images, predictions_list, num_cols=2, figsize=(15, 10)):
    """
    Create a composite figure with multiple images and predictions - masks only
    
    Args:
        images: List of images
        predictions_list: List of predictions for each image
        num_cols: Number of columns in the grid
        figsize: Figure dimensions
    """
    num_images = len(images)
    num_rows = (num_images + num_cols - 1) // num_cols
    
    fig, axes = plt.subplots(num_rows, num_cols, figsize=figsize)
    
    # Make axes always a 2D array even with a single row
    if num_rows == 1 and num_cols == 1:
        axes = np.array([[axes]])
    elif num_rows == 1:
        axes = np.array([axes])
    elif num_cols == 1:
        axes = axes.reshape(-1, 1)
    
    for i, (img, preds) in enumerate(zip(images, predictions_list)):
        row, col = i // num_cols, i % num_cols
        ax = axes[row, col]
        
        # Convert img to numpy array if needed
        if isinstance(img, torch.Tensor):
            img_np = img.cpu().numpy().transpose(1, 2, 0)
            if img_np.max() <= 1.0:
                img_np = (img_np * 255).astype(np.uint8)
        elif isinstance(img, np.ndarray):
            img_np = img.copy()
        else:
            img_np = np.array(img)
        
        # Show image
        ax.imshow(img_np)
        
        # Add masks
        masks = preds["masks"]
        
        # Prepare combined mask
        h, w = img_np.shape[:2]
        combined_mask = np.zeros((h, w, 4), dtype=np.uint8)
        
        if len(masks.shape) > 2:
            for j in range(len(masks)):
                mask = masks[j]
                mask_binary = (mask > 0.5).astype(np.uint8)
                
                # Color the mask
                r, g, b = BLUE_MASK_COLOR
                mask_color = np.zeros((h, w, 4), dtype=np.uint8)
                mask_color[mask_binary > 0] = [r, g, b, int(255 * MASK_ALPHA)]
                
                # Combine with existing mask
                combined_mask = np.maximum(combined_mask, mask_color)
            
            # Overlay mask
            mask_img = Image.fromarray(combined_mask, 'RGBA')
            ax.imshow(mask_img)
        
        ax.set_title(f"Frame {i}: {len(preds['boxes'])} objects", fontsize=12)
        ax.axis('off')
    
    # Hide empty axes
    for i in range(num_images, num_rows * num_cols):
        row, col = i // num_cols, i % num_cols
        axes[row, col].axis('off')
    
    plt.tight_layout()
    return fig

def create_comparison_figure(image, predictions, gt_boxes=None, figsize=(12, 6), show_boxes=False):
    """
    Create a comparison figure with predictions - masks only
    
    Args:
        image: Original image
        predictions: Model predictions
        gt_boxes: Ground truth bounding boxes (optional)
        figsize: Figure dimensions
        show_boxes: Whether to show ground truth boxes
    """
    # Convert image to numpy array if needed
    if isinstance(image, torch.Tensor):
        img_np = image.cpu().numpy().transpose(1, 2, 0)
        if img_np.max() <= 1.0:
            img_np = (img_np * 255).astype(np.uint8)
    elif isinstance(image, np.ndarray):
        img_np = image.copy()
    else:
        img_np = np.array(image)
    
    # Create figure with two side-by-side panels
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)
    
    # Left panel: original image
    ax1.imshow(img_np)
    ax1.set_title("Original image", fontsize=14)
    ax1.axis('off')
    
    # Right panel: image with masks
    ax2.imshow(img_np)
    
    # Add masks
    masks = predictions["masks"]
    
    # Prepare combined mask
    h, w = img_np.shape[:2]
    combined_mask = np.zeros((h, w, 4), dtype=np.uint8)
    
    if len(masks.shape) > 2:
        for j in range(len(masks)):
            mask = masks[j]
            mask_binary = (mask > 0.5).astype(np.uint8)
            
            # Color the mask
            r, g, b = BLUE_MASK_COLOR
            mask_color = np.zeros((h, w, 4), dtype=np.uint8)
            mask_color[mask_binary > 0] = [r, g, b, int(255 * MASK_ALPHA)]
            
            # Combine with existing mask
            combined_mask = np.maximum(combined_mask, mask_color)
        
        # Overlay mask
        mask_img = Image.fromarray(combined_mask, 'RGBA')
        ax2.imshow(mask_img)
    
    # Add ground truth boxes if available and desired
    if gt_boxes is not None and show_boxes:
        green_color_norm = GREEN_CONTOUR_COLOR_NORM.copy()
        for box in gt_boxes:
            x1, y1, x2, y2 = box.astype(int)
            rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, 
                               edgecolor=green_color_norm, linewidth=2, linestyle='--')
            ax2.add_patch(rect)
    
    ax2.set_title(f"In-hand object segmentation ({len(predictions['boxes'])} detected)", fontsize=14)
    ax2.axis('off')
    
    plt.tight_layout()
    return fig

def plot_training_history(history):
    """Plot training history metrics"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Plot training loss
    epochs = range(1, len(history['loss']) + 1)
    ax1.plot(epochs, history['loss'], 'b-', label='Training loss')
    if history['val_loss'] is not None:
        ax1.plot(epochs, history['val_loss'], 'r-', label='Validation loss')
    ax1.set_title('Training and Validation Loss')
    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot learning rate
    ax2.plot(epochs, history['lr'], 'g-')
    ax2.set_title('Learning Rate')
    ax2.set_xlabel('Epochs')
    ax2.set_ylabel('Learning Rate')
    ax2.set_yscale('log')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    return fig

## 5. Data Collection and Preparation

Let's collect all the available sequences and prepare our dataset.

In [7]:
def get_all_sequences(dataset_path):
    """Get all available Hot3D sequences"""
    sequences = []
    for item in os.listdir(dataset_path):
        item_path = os.path.join(dataset_path, item)
        # Skip the assets folder and non-directories
        if item == "assets" or not os.path.isdir(item_path):
            continue
        # Check if this is a valid Hot3D sequence (has recording.vrs file)
        vrs_path = os.path.join(item_path, "recording.vrs")
        if os.path.exists(vrs_path):
            sequences.append(item_path)
    
    print(f"Found {len(sequences)} Hot3D sequences")
    return sequences

# Get all sequences
all_sequences = get_all_sequences(hot3d_dataset_path)

# Split into train, validation, and test sets (70% train, 15% val, 15% test)
train_seqs, temp_seqs = train_test_split(all_sequences, test_size=0.3, random_state=42)
val_seqs, test_seqs = train_test_split(temp_seqs, test_size=0.5, random_state=42)

print(f"Train sequences: {len(train_seqs)}")
print(f"Validation sequences: {len(val_seqs)}")
print(f"Test sequences: {len(test_seqs)}")

# Print some examples
print("\nExample train sequences:")
for seq in train_seqs[:3]:
    print(f"  - {os.path.basename(seq)}")

print("\nExample validation sequences:")
for seq in val_seqs[:3]:
    print(f"  - {os.path.basename(seq)}")

print("\nExample test sequences:")
for seq in test_seqs[:3]:
    print(f"  - {os.path.basename(seq)}")

Found 104 Hot3D sequences
Train sequences: 72
Validation sequences: 16
Test sequences: 16

Example train sequences:
  - P0012_6e4f7815
  - P0010_e481fd15
  - P0003_f4730606

Example validation sequences:
  - P0009_e27ad19f
  - P0009_cf323827
  - P0001_f71fc9b1

Example test sequences:
  - P0001_9b6feab7
  - P0001_550ea2ac
  - P0005_f838e862


## 6. Training the Mask R-CNN Model

Now, let's create datasets and train our model.

In [14]:
print(f"Using {len(train_seqs)} training sequences and {len(val_seqs)} validation sequences")

# Create datasets
train_dataset = Hot3DHandObjectDataset(
    sequence_paths=train_seqs,
    num_samples_per_sequence=20,  # Samples per sequence
    filter_empty=True,            # Skip frames without objects
    include_hands=True           # Don't include hands as a class
)

val_dataset = Hot3DHandObjectDataset(
    sequence_paths=val_seqs,
    num_samples_per_sequence=10,  
    filter_empty=True,            
    include_hands=True           
)

Using 10 training sequences and 3 validation sequences


Loading sequences:   0%|                                                                                                                    | 0/10 [00:00<?, ?it/s]

No MANO hand model provided, skipping MANO hand data provider


[MultiRecordFileReader][DEBUG]: Opened file '/storage/aspoto/hot3d/hot3d/dataset/P0012_6e4f7815/recording.vrs' and assigned to reader #0
[VrsDataProvider][INFO]: streamId 214-1/camera-rgb activated
[VrsDataProvider][WARNING]: Unsupported TimeSync mode: APP, ignoring.
[VrsDataProvider][INFO]: Timecode stream found: 285-2
[VrsDataProvider][INFO]: Fail to activate streamId 286-1
[VrsDataProvider][INFO]: streamId 1201-1/camera-slam-left activated
[VrsDataProvider][INFO]: streamId 1201-2/camera-slam-right activated
[VrsDataProvider][INFO]: streamId 1202-1/imu-right activated
[VrsDataProvider][INFO]: streamId 1202-2/imu-left activated
[MpsDataPathsProvider][WARNING]: Hand tracking folder (/storage/aspoto/hot3d/hot3d/dataset/P0012_6e4f7815/mps/hand_tracking) does not exist in MPS root folder, not loading wrist and palm poses.


MPS Data Paths
MPS SLAM Data Paths
--closedLoopTrajectory: /storage/aspoto/hot3d/hot3d/dataset/P0012_6e4f7815/mps/slam/closed_loop_trajectory.csv
--openLoopTrajectory: /storage/aspoto/hot3d/hot3d/dataset/P0012_6e4f7815/mps/slam/open_loop_trajectory.csv
--semidensePoints: /storage/aspoto/hot3d/hot3d/dataset/P0012_6e4f7815/mps/slam/semidense_points.csv.gz
--semidenseObservations: /storage/aspoto/hot3d/hot3d/dataset/P0012_6e4f7815/mps/slam/semidense_observations.csv.gz
--onlineCalibration: /storage/aspoto/hot3d/hot3d/dataset/P0012_6e4f7815/mps/slam/online_calibration.jsonl
--summary: /storage/aspoto/hot3d/hot3d/dataset/P0012_6e4f7815/mps/slam/summary.json
MPS Eyegaze Data Paths
--generalEyegaze: /storage/aspoto/hot3d/hot3d/dataset/P0012_6e4f7815/mps/eye_gaze/general_eye_gaze.csv
--personalizedEyegaze: /storage/aspoto/hot3d/hot3d/dataset/P0012_6e4f7815/mps/eye_gaze/personalized_eye_gaze.csv
--summary: /storage/aspoto/hot3d/hot3d/dataset/P0012_6e4f7815/mps/eye_gaze/summary.json
MPS Hand Tra

...

No MANO hand model provided, skipping MANO hand data provider


[MultiRecordFileReader][DEBUG]: Opened file '/storage/aspoto/hot3d/hot3d/dataset/P0001_f71fc9b1/recording.vrs' and assigned to reader #0
[VrsDataProvider][INFO]: streamId 214-1/camera-rgb activated
[VrsDataProvider][WARNING]: Unsupported TimeSync mode: APP, ignoring.
[VrsDataProvider][INFO]: Timecode stream found: 285-2
[VrsDataProvider][INFO]: Fail to activate streamId 286-1
[VrsDataProvider][INFO]: streamId 1201-1/camera-slam-left activated
[VrsDataProvider][INFO]: streamId 1201-2/camera-slam-right activated
[VrsDataProvider][INFO]: streamId 1202-1/imu-right activated
[VrsDataProvider][INFO]: streamId 1202-2/imu-left activated
[MpsDataPathsProvider][WARNING]: Hand tracking folder (/storage/aspoto/hot3d/hot3d/dataset/P0001_f71fc9b1/mps/hand_tracking) does not exist in MPS root folder, not loading wrist and palm poses.


MPS Data Paths
MPS SLAM Data Paths
--closedLoopTrajectory: /storage/aspoto/hot3d/hot3d/dataset/P0001_f71fc9b1/mps/slam/closed_loop_trajectory.csv
--openLoopTrajectory: /storage/aspoto/hot3d/hot3d/dataset/P0001_f71fc9b1/mps/slam/open_loop_trajectory.csv
--semidensePoints: /storage/aspoto/hot3d/hot3d/dataset/P0001_f71fc9b1/mps/slam/semidense_points.csv.gz
--semidenseObservations: /storage/aspoto/hot3d/hot3d/dataset/P0001_f71fc9b1/mps/slam/semidense_observations.csv.gz
--onlineCalibration: /storage/aspoto/hot3d/hot3d/dataset/P0001_f71fc9b1/mps/slam/online_calibration.jsonl
--summary: /storage/aspoto/hot3d/hot3d/dataset/P0001_f71fc9b1/mps/slam/summary.json
MPS Eyegaze Data Paths
--generalEyegaze: /storage/aspoto/hot3d/hot3d/dataset/P0001_f71fc9b1/mps/eye_gaze/general_eye_gaze.csv
--personalizedEyegaze: /storage/aspoto/hot3d/hot3d/dataset/P0001_f71fc9b1/mps/eye_gaze/personalized_eye_gaze.csv
--summary: /storage/aspoto/hot3d/hot3d/dataset/P0001_f71fc9b1/mps/eye_gaze/summary.json
MPS Hand Tra

Loading sequences: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:07<00:00,  2.47s/it]

Dataset prepared with 30 samples


In [19]:
# Create the model handler
handler = MaskRCNNHandler(num_classes=2, pretrained=True)  # 2 classes: background and object

# Settings for training
num_epochs = 10      # Number of training epochs
batch_size = 2       # Batch size
learning_rate = 0.001  # Learning rate

# Model save path
model_save_path = os.path.join(models_dir, "maskrcnn_inhand_objects.pth")

# Train the model
history = handler.train(
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    num_epochs=num_epochs,
    batch_size=batch_size,
    lr=learning_rate
)

# Save the model
handler.save_model(model_save_path)

# Save training history
history_path = os.path.join(models_dir, "training_history.json")
with open(history_path, 'w') as f:
    json.dump(history, f)

print(f"Model saved to {model_save_path}")
print(f"Training history saved to {history_path}")

# Plot training history
fig = plot_training_history(history)
history_plot_path = os.path.join(visualizations_dir, "training_history.png")
fig.savefig(history_plot_path, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f"Training history plot saved to {history_plot_path}")

Using device: cuda
Starting training...


Validation: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 15/15 [00:05<00:00,  2.86it/s]


Epoch 1/10, Loss: 1.4351, Val Loss: 1607.5408, LR: 0.001000, Time: 25.20s
Model saved to /storage/aspoto/hot3d/hot3d/mask_rcnn_project/models/maskrcnn_inhand_objects_epoch_0.pth


Validation: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 15/15 [00:03<00:00,  3.85it/s]


Epoch 2/10, Loss: 0.8612, Val Loss: 1628.6661, LR: 0.001000, Time: 24.17s
Model saved to /storage/aspoto/hot3d/hot3d/mask_rcnn_project/models/maskrcnn_inhand_objects_epoch_1.pth


...

Epoch 8/10, Loss: 0.5530, Val Loss: 1623.3142, LR: 0.000010, Time: 25.82s
Model saved to /storage/aspoto/hot3d/hot3d/mask_rcnn_project/models/maskrcnn_inhand_objects_epoch_7.pth


Validation: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 15/15 [00:04<00:00,  3.71it/s]


Epoch 9/10, Loss: 0.5486, Val Loss: 1627.3753, LR: 0.000001, Time: 26.41s
Model saved to /storage/aspoto/hot3d/hot3d/mask_rcnn_project/models/maskrcnn_inhand_objects_epoch_8.pth


Validation: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 15/15 [00:04<00:00,  3.49it/s]


Epoch 10/10, Loss: 0.5479, Val Loss: 1627.3385, LR: 0.000001, Time: 25.23s
Model saved to /storage/aspoto/hot3d/hot3d/mask_rcnn_project/models/maskrcnn_inhand_objects_epoch_9.pth
Training completed!
Model saved to /storage/aspoto/hot3d/hot3d/mask_rcnn_project/models/maskrcnn_inhand_objects.pth
Model saved to /storage/aspoto/hot3d/hot3d/mask_rcnn_project/models/maskrcnn_inhand_objects.pth
Training history saved to /storage/aspoto/hot3d/hot3d/mask_rcnn_project/models/training_history.json
Training history plot saved to /storage/aspoto/hot3d/hot3d/mask_rcnn_project/visualizations/training_history.png


## 7. Model Evaluation

Let's evaluate our trained model on the test dataset.

In [20]:
print(f"Using {len(test_seqs)} test sequences")

test_dataset = Hot3DHandObjectDataset(
    sequence_paths=test_seqs,
    num_samples_per_sequence=20,  # Samples per sequence
    filter_empty=True,            # Skip frames without objects
    include_hands=True           # Don't include hands as a class
)

# Load model if not already loaded
if not os.path.exists(model_save_path):
    print(f"Model file {model_save_path} not found. Please train the model first.")
else:
    handler.load_model(model_save_path)

# Evaluate model
metrics = handler.evaluate_model(test_dataset)

# Save metrics
metrics_path = os.path.join(results_dir, "evaluation_metrics.json")
with open(metrics_path, 'w') as f:
    # Convert any numpy values to Python native types for JSON serialization
    json_metrics = {}
    for k, v in metrics.items():
        if isinstance(v, np.ndarray):
            json_metrics[k] = v.tolist()
        elif isinstance(v, np.floating):
            json_metrics[k] = float(v)
        elif isinstance(v, np.integer):
            json_metrics[k] = int(v)
        else:
            json_metrics[k] = v
    json.dump(json_metrics, f, indent=2)

print(f"Evaluation metrics saved to {metrics_path}")

# Create bar plot comparing predicted vs ground truth object counts
plt.figure(figsize=(12, 6))
indices = np.arange(len(metrics['detection_count']))
width = 0.35

plt.bar(indices - width/2, metrics['detection_count'], width, label='Predicted')
plt.bar(indices + width/2, metrics['gt_count'], width, label='Ground Truth')

plt.xlabel('Sample Index')
plt.ylabel('Number of Objects')
plt.title('Comparison of Predicted vs Ground Truth Object Counts')
plt.legend()
plt.grid(True, alpha=0.3)

# Save the plot
comparison_plot_path = os.path.join(visualizations_dir, "prediction_vs_gt_counts.png")
plt.savefig(comparison_plot_path, dpi=150, bbox_inches='tight')
plt.close()
print(f"Comparison plot saved to {comparison_plot_path}")

Using 5 test sequences


Loading sequences:   0%|                                                                                                                     | 0/5 [00:00<?, ?it/s]

No MANO hand model provided, skipping MANO hand data provider


[MultiRecordFileReader][DEBUG]: Opened file '/storage/aspoto/hot3d/hot3d/dataset/P0001_9b6feab7/recording.vrs' and assigned to reader #0
[VrsDataProvider][INFO]: streamId 214-1/camera-rgb activated
[VrsDataProvider][WARNING]: Unsupported TimeSync mode: APP, ignoring.
[VrsDataProvider][INFO]: Timecode stream found: 285-2
[VrsDataProvider][INFO]: Fail to activate streamId 286-1
[VrsDataProvider][INFO]: streamId 1201-1/camera-slam-left activated
[VrsDataProvider][INFO]: streamId 1201-2/camera-slam-right activated
[VrsDataProvider][INFO]: streamId 1202-1/imu-right activated
[VrsDataProvider][INFO]: streamId 1202-2/imu-left activated
[MpsDataPathsProvider][WARNING]: Hand tracking folder (/storage/aspoto/hot3d/hot3d/dataset/P0001_9b6feab7/mps/hand_tracking) does not exist in MPS root folder, not loading wrist and palm poses.


MPS Data Paths
MPS SLAM Data Paths
--closedLoopTrajectory: /storage/aspoto/hot3d/hot3d/dataset/P0001_9b6feab7/mps/slam/closed_loop_trajectory.csv
--openLoopTrajectory: /storage/aspoto/hot3d/hot3d/dataset/P0001_9b6feab7/mps/slam/open_loop_trajectory.csv
--semidensePoints: /storage/aspoto/hot3d/hot3d/dataset/P0001_9b6feab7/mps/slam/semidense_points.csv.gz
--semidenseObservations: /storage/aspoto/hot3d/hot3d/dataset/P0001_9b6feab7/mps/slam/semidense_observations.csv.gz
--onlineCalibration: /storage/aspoto/hot3d/hot3d/dataset/P0001_9b6feab7/mps/slam/online_calibration.jsonl
--summary: /storage/aspoto/hot3d/hot3d/dataset/P0001_9b6feab7/mps/slam/summary.json
MPS Eyegaze Data Paths
--generalEyegaze: /storage/aspoto/hot3d/hot3d/dataset/P0001_9b6feab7/mps/eye_gaze/general_eye_gaze.csv
--personalizedEyegaze: /storage/aspoto/hot3d/hot3d/dataset/P0001_9b6feab7/mps/eye_gaze/personalized_eye_gaze.csv
--summary: /storage/aspoto/hot3d/hot3d/dataset/P0001_9b6feab7/mps/eye_gaze/summary.json
MPS Hand Tra

...

MPS Data Paths
MPS SLAM Data Paths
--closedLoopTrajectory: /storage/aspoto/hot3d/hot3d/dataset/P0001_9c030609/mps/slam/closed_loop_trajectory.csv
--openLoopTrajectory: /storage/aspoto/hot3d/hot3d/dataset/P0001_9c030609/mps/slam/open_loop_trajectory.csv
--semidensePoints: /storage/aspoto/hot3d/hot3d/dataset/P0001_9c030609/mps/slam/semidense_points.csv.gz
--semidenseObservations: /storage/aspoto/hot3d/hot3d/dataset/P0001_9c030609/mps/slam/semidense_observations.csv.gz
--onlineCalibration: /storage/aspoto/hot3d/hot3d/dataset/P0001_9c030609/mps/slam/online_calibration.jsonl
--summary: /storage/aspoto/hot3d/hot3d/dataset/P0001_9c030609/mps/slam/summary.json
MPS Eyegaze Data Paths
--generalEyegaze: /storage/aspoto/hot3d/hot3d/dataset/P0001_9c030609/mps/eye_gaze/general_eye_gaze.csv
--personalizedEyegaze: /storage/aspoto/hot3d/hot3d/dataset/P0001_9c030609/mps/eye_gaze/personalized_eye_gaze.csv
--summary: /storage/aspoto/hot3d/hot3d/dataset/P0001_9c030609/mps/eye_gaze/summary.json
MPS Hand Tra

Loading sequences: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [02:28<00:00, 29.76s/it]
/tmp/ipykernel_95824/4231222347.py:277: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please op

Dataset prepared with 43 samples
Model loaded from /storage/aspoto/hot3d/hot3d/mask_rcnn_project/models/maskrcnn_inhand_objects.pth


Evaluating: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 22/22 [00:04<00:00,  5.10it/s]


Evaluation Results:
Total predicted objects: 220
Total ground truth objects: 203
Average difference per frame: 1.65
Evaluation metrics saved to /storage/aspoto/hot3d/hot3d/mask_rcnn_project/results/evaluation_metrics.json
Comparison plot saved to /storage/aspoto/hot3d/hot3d/mask_rcnn_project/visualizations/prediction_vs_gt_counts.png


## 8. Inference and Visualization on Test Images

Now let's run inference on some test images and visualize the results.

In [31]:
# Select a few test images for visualization
num_samples = 8
test_indices = np.linspace(0, len(test_dataset)-1, num_samples, dtype=int)
test_images = []
test_targets = []
test_predictions = []

for idx in test_indices:
    image, target = test_dataset[idx]
    
    # Ensure the image is in the correct format for prediction
    # Convert tensor to numpy for prediction
    if isinstance(image, torch.Tensor):
        # Convert tensor to numpy, transpose if needed (from CxHxW to HxWxC)
        if image.ndim == 3:
            # Normalize to 0-255 range and convert to uint8 if it's float
            if image.dtype == torch.float32:
                image_for_pred = (image.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
            else:
                image_for_pred = image.permute(1, 2, 0).numpy()
        else:
            # For batched tensor, squeeze and handle similarly
            if image.dtype == torch.float32:
                image_for_pred = (image.squeeze(0).permute(1, 2, 0).numpy() * 255).astype(np.uint8)
            else:
                image_for_pred = image.squeeze(0).permute(1, 2, 0).numpy()
    else:
        # If already numpy, ensure it's in HxWxC format and uint8
        if image.dtype == np.float32:
            image_for_pred = (image * 255).astype(np.uint8)
        else:
            image_for_pred = image

    # Ensure the image is uint8 and has correct shape
    if image_for_pred.dtype != np.uint8:
        image_for_pred = image_for_pred.astype(np.uint8)
    
    # Ensure 3 color channels
    if image_for_pred.ndim == 2:
        image_for_pred = np.stack([image_for_pred]*3, axis=-1)
    elif image_for_pred.shape[2] == 1:
        image_for_pred = np.repeat(image_for_pred, 3, axis=2)

    # Run inference
    prediction = handler.predict(image_for_pred, score_threshold=0.5)
    
    test_images.append(image)
    test_targets.append(target)
    test_predictions.append(prediction)

# Create a directory for individual visualizations
individual_viz_dir = os.path.join(visualizations_dir, "individual_samples")
os.makedirs(individual_viz_dir, exist_ok=True)

# Generate and save individual visualizations
for i, (image, target, prediction) in enumerate(zip(test_images, test_targets, test_predictions)):
    # Create visualization
    fig = visualize_results(image, prediction)
    
    # Save the figure
    fig_path = os.path.join(individual_viz_dir, f"sample_{i:02d}_maskrcnn.png")
    fig.savefig(fig_path, bbox_inches='tight', dpi=150)
    plt.close(fig)
    
    # Create comparison visualization
    # Convert target['boxes'] from tensor to numpy array if needed
    gt_boxes = target['boxes'].numpy() if isinstance(target['boxes'], torch.Tensor) else target['boxes']
    
    fig_comp = create_comparison_figure(image, prediction, gt_boxes)
    fig_comp_path = os.path.join(individual_viz_dir, f"sample_{i:02d}_comparison.png")
    fig_comp.savefig(fig_comp_path, bbox_inches='tight', dpi=150)
    plt.close(fig_comp)

# Create and save a composite visualization
composite_fig = create_composite_figure(test_images, test_predictions)
composite_path = os.path.join(visualizations_dir, "composite_results.png")
composite_fig.savefig(composite_path, bbox_inches='tight', dpi=200)
plt.close(composite_fig)

print(f"Individual visualizations saved to {individual_viz_dir}")
print(f"Composite visualization saved to {composite_path}")

Individual visualizations saved to /storage/aspoto/hot3d/hot3d/mask_rcnn_project/visualizations/individual_samples
Composite visualization saved to /storage/aspoto/hot3d/hot3d/mask_rcnn_project/visualizations/composite_results.png


## 9. Process a Complete Sequence

Finally, let's process a complete sequence and create a visualization.

In [32]:
def process_sequence(sequence_path, handler, output_dir, num_frames=10):
    """Process a Hot3D sequence and generate visualizations"""
    # Path to object library
    object_library_path = os.path.join(hot3d_dataset_path, "assets")
    
    # Load object library
    object_library = load_object_library(object_library_folderpath=object_library_path)
    
    # Initialize data provider
    hot3d_data_provider = Hot3dDataProvider(
        sequence_folder=sequence_path,
        object_library=object_library
    )
    
    # Get device data provider
    device_data_provider = hot3d_data_provider.device_data_provider
    
    # Get object bounding box data provider for ground truth
    object_box2d_data_provider = hot3d_data_provider.object_box2d_data_provider
    
    # Get timestamps
    timestamps = device_data_provider.get_sequence_timestamps()
    
    # Get image stream IDs
    image_stream_ids = device_data_provider.get_image_stream_ids()
    
    # Select RGB stream if available, otherwise use SLAM-LEFT
    if StreamId("214-1") in image_stream_ids:
        stream_id = StreamId("214-1")  # RGB
        print("Using RGB stream")
    else:
        stream_id = StreamId("1201-1")  # SLAM-LEFT
        print("Using SLAM-LEFT stream")
    
    # Sample frames evenly across the sequence
    frame_indices = np.linspace(0, len(timestamps)-1, num_frames, dtype=int)
    
    # Lists to store images and predictions
    all_images = []
    all_predictions = []
    
    # Process frames
    for idx, frame_idx in enumerate(tqdm(frame_indices, desc="Processing frames")):
        if frame_idx < len(timestamps):
            timestamp_ns = timestamps[frame_idx]
            
            # Get image
            image_data = device_data_provider.get_image(timestamp_ns, stream_id)
            
            # Run inference
            predictions = handler.predict(image_data, score_threshold=0.5)
            
            # Store images and predictions for composite figure
            all_images.append(image_data)
            all_predictions.append(predictions)
            
            # Generate individual visualization
            fig = visualize_results(image_data, predictions)
            fig.savefig(os.path.join(output_dir, f"frame_{idx:04d}.png"), 
                       bbox_inches='tight', dpi=150)
            plt.close(fig)
            
            # Get ground truth if available
            gt_boxes = None
            if object_box2d_data_provider is not None:
                box2d_collection_with_dt = object_box2d_data_provider.get_bbox_at_timestamp(
                    stream_id=stream_id,
                    timestamp_ns=timestamp_ns,
                    time_query_options=TimeQueryOptions.CLOSEST,
                    time_domain=TimeDomain.TIME_CODE,
                )
                
                if box2d_collection_with_dt is not None:
                    # Extract object bounding boxes
                    gt_boxes = []
                    for obj_uid in box2d_collection_with_dt.box2d_collection.object_uid_list:
                        bbox_data = box2d_collection_with_dt.box2d_collection.box2ds[obj_uid]
                        if bbox_data.box2d is not None:
                            # Convert box2d to [x1, y1, x2, y2] format
                            bb = bbox_data.box2d
                            gt_boxes.append(np.array([bb.left, bb.top, bb.left + bb.width, bb.top + bb.height]))
                    
                    # Convert to numpy array if there are any boxes
                    if gt_boxes:
                        gt_boxes = np.array(gt_boxes)
            
            # Generate comparison visualization
            fig_comp = create_comparison_figure(image_data, predictions, gt_boxes, show_boxes=True)
            fig_comp.savefig(os.path.join(output_dir, f"comparison_{idx:04d}.png"), 
                           bbox_inches='tight', dpi=150)
            plt.close(fig_comp)
    
    # Create composite visualization
    fig_grid = create_composite_figure(all_images, all_predictions, num_cols=4)
    fig_grid.savefig(os.path.join(output_dir, f"sequence_overview.png"), 
                    bbox_inches='tight', dpi=200)
    plt.close(fig_grid)
    
    print(f"Sequence processing completed. Results saved to {output_dir}")

# Select a test sequence to process
test_sequence = test_seqs[0]
sequence_name = os.path.basename(test_sequence)

# Create output directory
sequence_output_dir = os.path.join(results_dir, f"sequence_{sequence_name}")
os.makedirs(sequence_output_dir, exist_ok=True)

# Process the sequence
process_sequence(
    sequence_path=test_sequence,
    handler=handler,
    output_dir=sequence_output_dir,
    num_frames=16  # Process 16 frames from the sequence
)

No MANO hand model provided, skipping MANO hand data provider


[MultiRecordFileReader][DEBUG]: Opened file '/storage/aspoto/hot3d/hot3d/dataset/P0001_9b6feab7/recording.vrs' and assigned to reader #0
[VrsDataProvider][INFO]: streamId 214-1/camera-rgb activated
[VrsDataProvider][WARNING]: Unsupported TimeSync mode: APP, ignoring.
[VrsDataProvider][INFO]: Timecode stream found: 285-2
[VrsDataProvider][INFO]: Fail to activate streamId 286-1
[VrsDataProvider][INFO]: streamId 1201-1/camera-slam-left activated
[VrsDataProvider][INFO]: streamId 1201-2/camera-slam-right activated
[VrsDataProvider][INFO]: streamId 1202-1/imu-right activated
[VrsDataProvider][INFO]: streamId 1202-2/imu-left activated
[MpsDataPathsProvider][WARNING]: Hand tracking folder (/storage/aspoto/hot3d/hot3d/dataset/P0001_9b6feab7/mps/hand_tracking) does not exist in MPS root folder, not loading wrist and palm poses.


MPS Data Paths
MPS SLAM Data Paths
--closedLoopTrajectory: /storage/aspoto/hot3d/hot3d/dataset/P0001_9b6feab7/mps/slam/closed_loop_trajectory.csv
--openLoopTrajectory: /storage/aspoto/hot3d/hot3d/dataset/P0001_9b6feab7/mps/slam/open_loop_trajectory.csv
--semidensePoints: /storage/aspoto/hot3d/hot3d/dataset/P0001_9b6feab7/mps/slam/semidense_points.csv.gz
--semidenseObservations: /storage/aspoto/hot3d/hot3d/dataset/P0001_9b6feab7/mps/slam/semidense_observations.csv.gz
--onlineCalibration: /storage/aspoto/hot3d/hot3d/dataset/P0001_9b6feab7/mps/slam/online_calibration.jsonl
--summary: /storage/aspoto/hot3d/hot3d/dataset/P0001_9b6feab7/mps/slam/summary.json
MPS Eyegaze Data Paths
--generalEyegaze: /storage/aspoto/hot3d/hot3d/dataset/P0001_9b6feab7/mps/eye_gaze/general_eye_gaze.csv
--personalizedEyegaze: /storage/aspoto/hot3d/hot3d/dataset/P0001_9b6feab7/mps/eye_gaze/personalized_eye_gaze.csv
--summary: /storage/aspoto/hot3d/hot3d/dataset/P0001_9b6feab7/mps/eye_gaze/summary.json
MPS Hand Tra

Processing frames: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 16/16 [01:27<00:00,  5.45s/it]


Sequence processing completed. Results saved to /storage/aspoto/hot3d/hot3d/mask_rcnn_project/results/sequence_P0001_9b6feab7


## 10. Conclusion and Next Steps

We've successfully trained and evaluated a Mask R-CNN model for in-hand object segmentation on the Hot3D dataset. Here's a summary of our project:

1. **Data Preparation**:
   - Created a custom dataset for in-hand objects in Hot3D
   - Split available sequences into train, validation, and test sets
   - Handled the extraction of object masks and annotations

2. **Model Training**:
   - Fine-tuned a pre-trained Mask R-CNN model for our specific task
   - Monitored training progress with validation metrics
   - Saved the trained model for future use

3. **Evaluation**:
   - Evaluated the model on held-out test sequences
   - Analyzed performance metrics and visualization results

4. **Visualization**:
   - Created high-quality visualizations of the segmentation results
   - Generated both individual and composite visualizations
   - Processed full sequences to demonstrate the model's capabilities

### Next Steps

For future improvements, we could consider:

1. **Adding Depth Information**:
   - Implement the MRCNN-DA variant that incorporates depth maps from Depth Anything V2
   - Compare performance with and without depth information

2. **Improving Training**:
   - Train for more epochs with more data
   - Experiment with different data augmentation techniques
   - Try different learning rate schedules

3. **Refining the Model**:
   - Experiment with different backbone architectures
   - Implement instance segmentation with hand context
   - Explore class-specific segmentation (multiple object classes)

The code is organized to make these extensions straightforward to implement.

## 11. Summary of Project Structure

Here's an overview of the project structure that we've created:

```
/hot3d/hot3d/mask_rcnn_project/
├── models/                       # Trained models and training history
│   ├── maskrcnn_inhand_objects.pth   # Trained model weights
│   └── training_history.json         # Training metrics
├── results/                      # Evaluation results
│   ├── evaluation_metrics.json       # Performance metrics
│   └── sequence_*/                   # Results for individual sequences
└── visualizations/               # Visualization outputs
    ├── training_history.png          # Training loss and learning rate plots
    ├── prediction_vs_gt_counts.png   # Comparison of predictions vs ground truth
    ├── composite_results.png         # Grid of multiple result images
    └── individual_samples/           # Individual result visualizations
```